In [1]:
import pandas as pd
import json
import csv
from datetime import datetime

# Configurações de exibição do Pandas para facilitar o debug
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [2]:
import urllib.request
import tarfile
import os

# 1. URL corrigida com 'id_' no final do timestamp (padrão do Internet Archive para arquivo cru)
url = "https://web.archive.org/web/20210420235203id_/http://research.moodle.org/158/2/export.tar.gz"
arquivo_compactado = "export.tar.gz"
arquivo_verificacao = "export/mdl_logstore_standard_log.csv" 

def baixar_e_extrair_dados():
    if os.path.exists(arquivo_verificacao):
        print("✅ Os dados brutos já estão presentes na pasta. Pulando o download!")
        return

    print(f"📥 Iniciando o download do dataset gigante... (Isso pode demorar dependendo da conexão)")
    try:
        # 2. Criando um disfarce (User-Agent) para não sermos bloqueados
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        
        # 3. Baixando o arquivo em "pedacinhos" para não travar a memória do computador
        with urllib.request.urlopen(req) as response, open(arquivo_compactado, 'wb') as out_file:
            while chunk := response.read(8192):
                out_file.write(chunk)
                
        print("📦 Download real concluído! Iniciando a extração dos arquivos...")

        # Extrai o .tar.gz
        with tarfile.open(arquivo_compactado, "r:gz") as tar:
            tar.extractall(path=".")
        
        print("🧹 Extração finalizada! Deletando o arquivo compactado para liberar espaço...")
        os.remove(arquivo_compactado)
        
        print("🚀 Tudo pronto! O dataset está descompactado e pronto para o Pandas.")
        
    except Exception as e:
        print(f"❌ Ocorreu um erro durante o processo: {e}")
        # Limpa o arquivo corrompido para não atrapalhar a próxima tentativa
        if os.path.exists(arquivo_compactado):
            os.remove(arquivo_compactado)

# Executa a função
baixar_e_extrair_dados()

✅ Os dados brutos já estão presentes na pasta. Pulando o download!


In [3]:
# Mapeamento oficial: Evento do Moodle -> Verbo xAPI
MOODLE_TO_XAPI_VERBS = {
    r'\core\event\course_viewed': 'http://id.tincanapi.com/verb/viewed',
    r'\mod_quiz\event\attempt_started': 'http://adlnet.gov/expapi/verbs/launched',
    r'\mod_quiz\event\attempt_submitted': 'http://activitystrea.ms/schema/1.0/submit',
    r'\mod_quiz\event\attempt_reviewed': 'http://id.tincanapi.com/verb/reviewed',
    r'\mod_assign\event\assessable_submitted': 'http://activitystrea.ms/schema/1.0/submit',
    r'\core\event\course_completed': 'http://adlnet.gov/expapi/verbs/completed',
    r'\core\event\badge_awarded': 'http://adlnet.gov/expapi/verbs/earned',
    r'\core\event\user_loggedin': 'https://w3id.org/xapi/adl/verbs/logged-in',
}

def obter_verbo_xapi(eventname):
    # Fallback para 'interacted' se o evento não estiver mapeado
    return MOODLE_TO_XAPI_VERBS.get(eventname, 'http://adlnet.gov/expapi/verbs/interacted')

In [4]:
def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def construir_object_id(component, instance_id, course_id):
    nome_modulo = component.replace("mod_", "") if component.startswith("mod_") else component
    return f"http://seumoodle.com/mod/{nome_modulo}/view.php?id={instance_id}&course={course_id}"

In [5]:
import pandas as pd
import json
from datetime import datetime

def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def segundos_para_iso8601(segundos):
    """Converte segundos inteiros para o padrão de duração do xAPI (ISO 8601)"""
    m, s = divmod(int(segundos), 60)
    h, m = divmod(m, 60)
    iso = "PT"
    if h > 0: iso += f"{h}H"
    if m > 0: iso += f"{m}M"
    iso += f"{s}S"
    return iso

def transformar_logs_para_xapi(df_logs):
    statements = []
    memoria_inicio = {}
    
    print("Ordenando logs no tempo...")
    df_logs = df_logs.sort_values('timecreated')

    for _, row in df_logs.iterrows():
        userid = str(row.get('username', ''))
        if userid in ['', '0', 'nan', '-1']: continue
            
        eventname = row.get('eventname', '')
        timecreated = int(row.get('timecreated', 0))
        
        # === VALORES QUE ESTAMOS LENDO PARA OS NOMES ===
        courseid = str(row.get('courseid', '0'))
        instanceid = str(row.get('contextinstanceid', '0'))
        component = str(row.get('component', 'core'))
        # ===============================================
        
        chave_atividade = (userid, courseid, instanceid)

        # 1. Memória de Início
        if "started" in eventname or "viewed" in eventname:
            if chave_atividade not in memoria_inicio: 
                memoria_inicio[chave_atividade] = timecreated

        statement = {
            "actor": {"objectType": "Agent", "account": {"homePage": "http://seumoodle.com", "name": userid}},
            "verb": {
                "id": obter_verbo_xapi(eventname),
                "display": {"en-US": row.get('action', 'interacted')}
            },
            "object": {
                "objectType": "Activity",
                "id": construir_object_id(component, instanceid, courseid),
                
                # ALTERAÇÃO 1: Nome da Atividade agora mostra o Componente + ID
                "definition": {"name": {"en-US": f"{component} (ID: {instanceid})"}}
            },
            "timestamp": formatar_timestamp(timecreated),
            "context": {
                "contextActivities": {
                    "parent": [
                        {
                            "id": f"http://seumoodle.com/course/view.php?id={courseid}",
                            "definition": {
                                "type": "activitytype/course",
                                
                                # ALTERAÇÃO 2: Nome da Matéria agora mostra o ID
                                "name": {"en": f"Matéria ID {courseid}"}
                            }
                        }
                    ]
                }
            }
        }

        # 2. Cálculo de Tempo de Resposta
        if "submitted" in eventname or "completed" in eventname:
            if chave_atividade in memoria_inicio:
                tempo_gasto_segundos = timecreated - memoria_inicio[chave_atividade]
                
                if tempo_gasto_segundos >= 0:
                    statement["result"] = {
                        "duration": segundos_para_iso8601(tempo_gasto_segundos)
                    }
                    statement["verb"]["id"] = "http://adlnet.gov/expapi/verbs/completed"
                    statement["verb"]["display"] = {"en-US": "completed"}
                
                del memoria_inicio[chave_atividade]

        statements.append(statement)
    return statements

def transformar_notas_para_xapi(df_notas):
    statements = []
    df_validas = df_notas.dropna(subset=['rawgrade', 'rawgrademax'])
    
    for _, row in df_validas.iterrows():
        userid = str(row.get('username', ''))
        if userid in ['', '0', 'nan', '-1']: continue

        item_id = str(row.get('itemid', '0'))
        courseid = str(row.get('courseid', item_id)) # Tenta pegar o courseid se existir
        
        statement = {
            "actor": {"objectType": "Agent", "account": {"name": userid}},
            "verb": {
                "id": "http://adlnet.gov/expapi/verbs/scored",
                "display": {"en-US": "scored"}
            },
            "object": {
                "objectType": "Activity",
                "id": f"http://seumoodle.com/grade/item/{item_id}"
            },
            "result": {
                "score": {
                    "raw": float(row['rawgrade']),
                    "max": float(row['rawgrademax'])
                }
            },
            "timestamp": formatar_timestamp(row.get('timemodified', 0)),
            "context": {
                "contextActivities": {
                    "parent": [
                        {
                            "id": f"http://seumoodle.com/mod/quiz/view.php?id={item_id}",
                            "definition": {
                                "type": "activitytype/course",
                                # ALTERAÇÃO 3: Garante o mesmo padrão nas notas
                                "name": {"en": f"Matéria ID {courseid}"}
                            }
                        }
                    ]
                }
            }
        }
        statements.append(statement)
    return statements

In [6]:
# 1. Carregamento dos Datasets
print("📖 Carregando arquivos originais...")
df_logs = pd.read_csv("export/mdl_logstore_standard_log.csv", low_memory=False)
df_notas = pd.read_csv("export/mdl_grade_grades_history.csv", low_memory=False)

# 2. Transformação
print(f"⚙️ Processando {len(df_logs)} logs de eventos...")
statements_logs = transformar_logs_para_xapi(df_logs)

print(f"⚙️ Processando {len(df_notas)} registros de notas...")
statements_notas = transformar_notas_para_xapi(df_notas)

# 3. União dos Statements
# Juntamos as duas listas em uma só
statements_totais = statements_logs + statements_notas

# 4. Exportação
caminho_output = "xapi_statements_completos.json"
print(f"💾 Salvando {len(statements_totais)} statements unificados em '{caminho_output}'...")

with open(caminho_output, "w", encoding="utf-8") as f:
    json.dump(statements_totais, f, indent=4, ensure_ascii=False)

print("🚀 Processo concluído! O arquivo está pronto para a sua métrica de pontuação.")

📖 Carregando arquivos originais...
⚙️ Processando 2391762 logs de eventos...
Ordenando logs no tempo...
⚙️ Processando 70037 registros de notas...
💾 Salvando 2400482 statements unificados em 'xapi_statements_completos.json'...
🚀 Processo concluído! O arquivo está pronto para a sua métrica de pontuação.


In [16]:
import shutil
import os

# Caminhos absolutos exatos (o 'r' na frente evita problemas com as barras do Windows)
origem = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\transformador\xapi_statements_completos.json'
destino_pasta = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data'
destino_arquivo = os.path.join(destino_pasta, 'xapi_statements_completos.json')

def mover_resultado():
    # Verificar se o arquivo de origem existe
    if not os.path.exists(origem):
        print(f"❌ Erro: O arquivo de origem não foi encontrado em:\n{origem}")
        return

    # Criar a pasta de destino se ela não existir
    if not os.path.exists(destino_pasta):
        os.makedirs(destino_pasta)
        print(f"📁 Pasta de destino criada: {destino_pasta}")

    try:
        # Mover o arquivo
        shutil.move(origem, destino_arquivo)
        print(f"✅ Sucesso! Arquivo movido para:\n{destino_arquivo}")
    except Exception as e:
        print(f"❌ Erro ao mover o arquivo: {e}")

mover_resultado()

✅ Sucesso! Arquivo movido para:
C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data\xapi_statements_completos.json


In [ ]:
import pandas as pd
import json
from datetime import datetime

path = "export/"

def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def segundos_para_iso8601(segundos):
    m, s = divmod(int(segundos), 60)
    h, m = divmod(m, 60)
    iso = "PT"
    if h > 0: iso += f"{h}H"
    if m > 0: iso += f"{m}M"
    iso += f"{s}S"
    return iso

def processar_logs_com_notas():
    print("📖 1. Carregando os dois arquivos...")
    df_logs = pd.read_csv(path +"mdl_logstore_standard_log.csv", low_memory=False)
    df_notas = pd.read_csv(path +"mdl_grade_grades_history.csv", low_memory=False)

    print("🧠 2. Criando o dicionário de busca rápida de notas...")
    # Prepara a "lista telefônica" de notas para a busca ser instantânea (O(1))
    df_validas = df_notas.dropna(subset=['rawgrade', 'rawgrademax'])
    dicionario_notas = {}
    
    for _, row in df_validas.iterrows():
        uid = str(row.get('username', ''))
        iid = str(row.get('itemid', '0'))
        if uid not in ['', '0', 'nan', '-1']:
            # Chave: (Usuário, ID da Atividade) -> Valor: Dicionário com as notas
            dicionario_notas[(uid, iid)] = {
                "raw": float(row['rawgrade']),
                "max": float(row['rawgrademax'])
            }

    print("⏱️ 3. Ordenando logs cronologicamente...")
    df_logs = df_logs.sort_values('timecreated')

    print(f"⚙️ 4. Varrendo {len(df_logs)} logs e buscando notas nos eventos de conclusão...")
    statements = []
    memoria_inicio = {}

    for _, row in df_logs.iterrows():
        userid = str(row.get('username', ''))
        if userid in ['', '0', 'nan', '-1']: continue

        eventname = row.get('eventname', '')
        timecreated = int(row.get('timecreated', 0))
        courseid = str(row.get('courseid', '0'))
        instanceid = str(row.get('contextinstanceid', '0'))
        component = str(row.get('component', 'core'))
        
        chave_atividade = (userid, courseid, instanceid)
        chave_busca_nota = (userid, instanceid) # A chave que usaremos para buscar a nota

        # Guarda o tempo de início
        if "started" in eventname or "viewed" in eventname:
            if chave_atividade not in memoria_inicio: 
                memoria_inicio[chave_atividade] = timecreated

        statement = {
            "actor": {"objectType": "Agent", "account": {"homePage": "http://seumoodle.com", "name": userid}},
            "verb": {
                "id": "http://id.tincanapi.com/verb/interacted", # O verbo padrão para cliques
                "display": {"en-US": row.get('action', 'interacted')}
            },
            "object": {
                "objectType": "Activity",
                "id": f"http://seumoodle.com/mod/{component}/view.php?id={instanceid}&course={courseid}",
                "definition": {"name": {"en-US": f"{component} (ID: {instanceid})"}}
            },
            "timestamp": formatar_timestamp(timecreated),
            "context": {
                "contextActivities": {
                    "parent": [
                        {
                            "id": f"http://seumoodle.com/course/view.php?id={courseid}",
                            "definition": {
                                "type": "activitytype/course",
                                "name": {"en": f"Matéria ID {courseid}"}
                            }
                        }
                    ]
                }
            }
        }

        # === A LÓGICA SOLICITADA ===
        # Se for um evento de Conclusão, buscamos o Tempo (memória) e a Nota (tabela)
        if "submitted" in eventname or "completed" in eventname:
            statement["verb"]["id"] = "http://adlnet.gov/expapi/verbs/completed"
            statement["verb"]["display"] = {"en-US": "completed"}
            statement["result"] = {}
            
            # 1. Puxa o Tempo
            if chave_atividade in memoria_inicio:
                tempo_gasto_segundos = timecreated - memoria_inicio[chave_atividade]
                if tempo_gasto_segundos >= 0:
                    statement["result"]["duration"] = segundos_para_iso8601(tempo_gasto_segundos)
                del memoria_inicio[chave_atividade]

            # 2. Puxa a Nota (Busca no Dicionário)
            if chave_busca_nota in dicionario_notas:
                statement["result"]["score"] = dicionario_notas[chave_busca_nota]

            # Se não achou nem tempo nem nota, apaga o bloco vazio
            if not statement["result"]:
                del statement["result"]

        statements.append(statement)

    # Exportação Final
    caminho_output = "xapi_statements_completos.json"
    print(f"💾 5. Salvando statements unificados em '{caminho_output}'...")
    with open(caminho_output, "w", encoding="utf-8") as f:
        json.dump(statements, f, indent=4, ensure_ascii=False)
        
    print("🚀 Sucesso! Arquivo gerado com a nova lógica.")

# Executar tudo
processar_logs_com_notas()

📖 1. Carregando os dois arquivos...
🧠 2. Criando o dicionário de busca rápida de notas...
⏱️ 3. Ordenando logs cronologicamente...
⚙️ 4. Varrendo 2391762 logs e buscando notas nos eventos de conclusão...
💾 5. Salvando statements unificados em 'xapi_statements_completos.json'...
🚀 Sucesso! Arquivo gerado com a nova lógica.
